In [1]:
# !pip install openai==0.28

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 2.4 MB/s eta 0:00:00


In [21]:
# ##uncomment this to use code in Google Colab
# !pip install fastchat
# !pip install wget
# !pip install evaluate
# !pip install sentence_transformers
# !pip install bert_score
# !pip install datasets

In [22]:
# ##uncomment this to use code in Google Colab
# from IPython.display import clear_output
# !git clone https://github.com/alfekka/lm-polygraph.git -b new_api
# %cd lm-polygraph
# %pip install .
# %cd src
# %pip install transformers rouge-score datasets openai
# !curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
# !apt-get install -y nodejs
# %cd lm_polygraph/app
# !npm install
# %cd ../../
# %cd /content/lm-polygraph/src
# %load_ext autoreload
# %autoreload 2

Cloning into 'lm-polygraph'...
remote: Enumerating objects: 6836, done.
remote: Counting objects: 100% (2946/2946), done.
remote: Compressing objects: 100% (780/780), done.
remote: Total 6836 (delta 2390), reused 2305 (delta 2077), pack-reused 3890
Receiving objects: 100% (6836/6836), 13.25 MiB | 19.39 MiB/s, done.
Resolving deltas: 100% (4092/4092), done.
/content/lm-polygraph/src/lm-polygraph
Processing /content/lm-polygraph/src/lm-polygraph
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 39.4 MB/s eta 0:00:00
  Using cached nlpaug-1.1.11-py3-none-any.whl (410 kB)
  Using cached bs4-0.0.2-py2.py3-none-any.whl (1.2 kB)
  Using cached transformers-4.36.0-py3-none-any.whl (8.2 MB)
  Using cached sacrebleu-2.4.2-py3-none-any.whl (106 kB)
  Using cached flask-3.0.3-py3-none-any.whl (101 kB)
  Using cach

In [1]:
# ##uncomment this to use code in Google Colab
# %cd ..
# !pwd
# %cd /content/lm-polygraph/src

/
/
[Errno 2] No such file or directory: '/content/lm-polygraph/src'
/


In [24]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [25]:
from lm_polygraph.generation_metrics.openai_fact_check import OpenAIFactCheck
from lm_polygraph.stat_calculators.extract_claims import Claim
import json
import pandas as pd
from tqdm import tqdm
russian_checker = OpenAIFactCheck('gpt-4o', language='ru')

In [26]:
from datasets import load_dataset
full = load_dataset("rvanova/person-bio-full")
full_claims = pd.DataFrame(full['test'])

In [27]:
full_claims.columns

Index(['question', 'bio', 'claim', 'sentence', 'tokens'], dtype='object')

In [28]:
for i, row in tqdm(full_claims.iterrows(), total=full_claims.shape[0]):
    claim = [Claim(claim_text=row['claim'], sentence=row['sentence'], aligned_tokens=row['tokens'])]
    check_dict = { "input_texts": [row['question']],
                  "claims": [claim]}
    chatgpt_response = russian_checker(check_dict, None, None )
    full_claims.at[i,'chatgpt_verdict'] = chatgpt_response[0]


100%|██████████| 80/80 [00:49<00:00,  1.63it/s]


In [31]:
%pwd

'/content/lm-polygraph/src'

In [32]:
full_claims.sample(frac=1).to_csv("../workdir/russian_fact_checked_claims_shuffled.csv", index=False)

full_claims.to_csv("../workdir/russian_fact_checked_claims.csv", index=False)